# 🔍 Lesson 02 — Oracle 23ai Vector Search
## Student Activity: Search a Wikipedia Article

In this activity you will:
1. **Choose** a Wikipedia article as your dataset
2. **Generate** vector embeddings for each paragraph (the "pre-load phase")
3. **Load** the vectors into Oracle 23ai on freesql.com
4. **Search** the article using natural language — no keyword matching needed

> This is exactly how Netflix, Spotify, and AI assistants find relevant content at scale.

## Step 1 — Install & Import

In [1]:
!pip install sentence-transformers wikipedia-api -q

from sentence_transformers import SentenceTransformer
import wikipediaapi
import re
import textwrap

print("Libraries loaded ✓")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 798.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.9/126.9 kB 2.6 MB/s eta 0:00:00
Libraries loaded ✓


## Step 2 — Choose Your Article

Pick one of the three articles below by setting `ARTICLE_CHOICE` to 1, 2, or 3:

| # | Article | Focus |
|---|---------|-------|
| 1 | Vector database | What they are, how they're built |
| 2 | Word embedding | How words become numbers |
| 3 | Semantic search | How meaning-based search works |

In [2]:
# ── CHANGE THIS: pick 1, 2, or 3 ──
ARTICLE_CHOICE = 3

ARTICLES = {
    1: "Vector database",
    2: "Word embedding",
    3: "Semantic search",
}

ARTICLE_TITLE = ARTICLES[ARTICLE_CHOICE]
print(f"You chose: '{ARTICLE_TITLE}'")

You chose: 'Semantic search'


## Step 3 — Fetch & Chunk the Article

In [3]:
wiki = wikipediaapi.Wikipedia(
    user_agent="oracle-vector-search-lesson/1.0",
    language="en"
)

page = wiki.page(ARTICLE_TITLE)
if not page.exists():
    raise ValueError(f"Article '{ARTICLE_TITLE}' not found on Wikipedia")

print(f"✓ Fetched: {page.title}")
print(f"  Length: {len(page.text):,} characters")

# Split into paragraphs, filter short/empty ones
raw_paragraphs = [p.strip() for p in page.text.split('\n') if len(p.strip()) > 120]

# Truncate to 400 chars max per chunk (fits Oracle VARCHAR2(2000) safely)
chunks = []
for i, para in enumerate(raw_paragraphs[:30]):   # cap at 30 chunks for this lesson
    chunk = para[:400]
    # Remove references like [1], [23]
    chunk = re.sub(r'\[\d+\]', '', chunk).strip()
    if len(chunk) > 80:
        chunks.append(chunk)

print(f"  Chunks: {len(chunks)}")
print()
print("Preview of first 3 chunks:")
for i, c in enumerate(chunks[:3]):
    print(f"  [{i+1}] {c[:100]}...")

✓ Fetched: Semantic search
  Length: 2,178 characters
  Chunks: 4

Preview of first 3 chunks:
  [1] Semantic search denotes search with meaning, as distinguished from lexical search where the search e...
  [2] Some authors regard semantic search as a set of techniques for retrieving knowledge from richly stru...
  [3] Semantic ontologies like Web Ontology Language, Resource Description Framework, and Schema.org organ...


## Step 4 — Generate Vector Embeddings

In [4]:
# Load the model (downloads ~90MB on first run)
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded ✓")

# Encode all chunks
embeddings = model.encode(chunks, show_progress_bar=True)
print(f"\nGenerated {len(embeddings)} embeddings, each with {len(embeddings[0])} dimensions")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:86: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded ✓


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Generated 4 embeddings, each with 384 dimensions


## Step 5 — Set Up the Database

📋 **Copy the SQL below and run it in freesql.com** (only once — it creates the table)

> freesql.com → SQL Workshop → SQL Commands → paste → Run

In [5]:
setup_sql = """-- ============================================================
-- Lesson 02 Step 5: Create the doc_chunks table
-- Run this in freesql.com BEFORE loading data
-- ============================================================

-- Drop if it already exists from a previous run
BEGIN
  EXECUTE IMMEDIATE 'DROP TABLE doc_chunks';
EXCEPTION WHEN OTHERS THEN NULL;
END;
/

CREATE TABLE doc_chunks (
    chunk_id     NUMBER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    doc_name     VARCHAR2(200),
    chunk_text   VARCHAR2(2000),
    chunk_vector VECTOR(384, FLOAT32)
);

-- Verify
SELECT table_name FROM user_tables WHERE table_name = 'DOC_CHUNKS';
"""

print(setup_sql)
print("=" * 60)
print("📋 Copy everything above and run it in freesql.com")

-- ============================================================
-- Lesson 02 Step 5: Create the doc_chunks table
-- Run this in freesql.com BEFORE loading data
-- ============================================================

-- Drop if it already exists from a previous run
BEGIN
  EXECUTE IMMEDIATE 'DROP TABLE doc_chunks';
EXCEPTION WHEN OTHERS THEN NULL;
END;
/

CREATE TABLE doc_chunks (
    chunk_id     NUMBER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    doc_name     VARCHAR2(200),
    chunk_text   VARCHAR2(2000),
    chunk_vector VECTOR(384, FLOAT32)
);

-- Verify
SELECT table_name FROM user_tables WHERE table_name = 'DOC_CHUNKS';

📋 Copy everything above and run it in freesql.com


## Step 6 — Generate INSERT Statements

Run this cell → copy the output → paste into freesql.com

> **Why the split trick?** Oracle rejects string literals over 4000 chars at parse time.
> A 384-dimension vector is ~4600 chars, so we split it in half using `TO_CLOB() || ...`

In [6]:
def format_vector(embedding):
    """Format embedding as Oracle VECTOR literal, split to avoid ORA-01704."""
    values = ", ".join(f"{v:.8f}" for v in embedding)
    full = f"[{values}]"
    mid = len(full) // 2
    split_pos = full.rindex(',', 0, mid) + 1
    part1 = full[:split_pos]
    part2 = full[split_pos:]
    return f"TO_VECTOR(TO_CLOB('{part1}') || '{part2}', 384, FLOAT32)"

print("-- ============================================================")
print(f"-- Lesson 02: Vector embeddings from '{ARTICLE_TITLE}'")
print("-- Run in freesql.com AFTER Step 5 (table must exist)")
print("-- ============================================================")
print()
for chunk, embedding in zip(chunks, embeddings):
    safe_text = chunk.replace("'", "''")[:390]   # stay under VARCHAR2(2000)
    vec = format_vector(embedding)
    print(f"INSERT INTO doc_chunks (doc_name, chunk_text, chunk_vector)")
    print(f"VALUES ('{ARTICLE_TITLE[:50]}', '{safe_text}', {vec});")
    print()
print("COMMIT;")
print()
print(f"-- Verify: {len(chunks)} rows expected")
print("SELECT COUNT(*) FROM doc_chunks;")

-- ============================================================
-- Lesson 02: Vector embeddings from 'Semantic search'
-- Run in freesql.com AFTER Step 5 (table must exist)
-- ============================================================

INSERT INTO doc_chunks (doc_name, chunk_text, chunk_vector)
VALUES ('Semantic search', 'Semantic search denotes search with meaning, as distinguished from lexical search where the search engine looks for literal matches of the query words or variants of them, without understanding the overall meaning of the query. Semantic search is an approach to information retrieval that seeks to improve search accuracy by understanding the searcher''s intent and the contextual meaning o', TO_VECTOR(TO_CLOB('[0.07394082, 0.05240205, -0.00850181, 0.01506318, -0.00603444, -0.00624556, 0.04179024, 0.00676755, -0.01350784, -0.02856040, 0.03712170, -0.03740408, 0.08689828, 0.01112024, 0.06928182, -0.00624566, 0.06573004, 0.02791443, 0.00064775, 0.03463808, 0.11610106, 0.

## Step 7 — Search Your Article

Change `MY_QUESTION` below, run the cell, copy the SQL, paste it into freesql.com.

In [10]:
MY_QUESTION = "What do Hybrid search models do?"
TOP_N = 3

q_emb = model.encode([MY_QUESTION])[0]
q_vec = format_vector(q_emb)

print(f"-- Search query: {MY_QUESTION}")
print(f"-- Top {TOP_N} most similar chunks")
print()
print("SELECT")
print("    chunk_id,")
print("    SUBSTR(chunk_text, 1, 100) AS preview,")
print(f"    ROUND(VECTOR_DISTANCE(chunk_vector, {q_vec}, COSINE), 4) AS similarity_score")
print("FROM doc_chunks")
print("ORDER BY similarity_score ASC")
print(f"FETCH FIRST {TOP_N} ROWS ONLY;")

-- Search query: What do Hybrid search models do?
-- Top 3 most similar chunks

SELECT
    chunk_id,
    SUBSTR(chunk_text, 1, 100) AS preview,
    ROUND(VECTOR_DISTANCE(chunk_vector, TO_VECTOR(TO_CLOB('[-0.04639228, -0.00308419, -0.02795119, -0.00462044, 0.02795612, 0.00772554, -0.04960556, -0.01674443, 0.03878120, -0.03515488, 0.04264085, 0.05147083, 0.06112472, -0.04645340, -0.00697198, -0.02363891, 0.05097563, -0.02276544, -0.05689747, -0.02392817, 0.01849512, -0.01555853, 0.00965241, -0.05622387, -0.08193725, -0.00733673, -0.02715480, -0.02055754, 0.02417182, -0.02751650, 0.01476415, 0.03044035, 0.05242160, 0.09201404, -0.03069908, -0.01916779, -0.07785282, -0.03233792, 0.02880737, -0.00917527, -0.05635468, -0.02412381, -0.00535063, -0.03391717, 0.06265011, -0.01678056, -0.04340805, -0.03316646, 0.03619957, -0.06612059, -0.07186634, -0.06294592, -0.03353803, 0.04386341, 0.05651249, 0.00553750, -0.00230193, -0.03629522, 0.00793601, -0.02751799, 0.06663284, -0.00379448, -0.05074812,

## 🎯 Activity — Your Turn

Try these three searches. For each one, run Step 7 with a new question, paste the SQL in freesql.com, and write down what you found.

---

**Search 1:** Ask something that IS in the article
> Example: `"What do Hybrid search models do?"`

What came back? Does it make sense?
Yes it makes sense, I got the answer I was looking for: Hybrid search models combine lexical retrieval (e.g., BM25) with semantic ranking using pretrained t

---

**Search 2:** Ask something that is RELATED but not a direct quote
> Example: `"What is semantic search?"`

Did it find relevant content even though those exact words aren't in the article?

Yes it did, it found things similar to this statement: Semantic search denotes search with meaning, as distinguished from lexical search where the search e...

---

**Search 3:** Ask something UNRELATED
> Example: `"how to make pasta"`

What score did you get? Is it high or low? Why?
The closest to meaning it found was a vector with 0.9925, which is really low, because the text does not include anything even remotely related to pasta

---

> 💡 **Key insight:** Vector search finds *meaning*, not keywords.
> A score near **0.0** = very similar. A score near **1.0** = very different.